# Quantum kernels
**Quantum Machine Learning · Week 2 (Quantum kernels) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qml/w2d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Put each data point into a quantum state $|\varphi(x)\rangle$ with a **feature-map circuit**, and use the
overlap $k(x, x') = |\langle\varphi(x)|\varphi(x')\rangle|^2$ (a fidelity!) as the kernel of an SVM.
That's a quantum support vector machine.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def encode_1q(x):
    qc = QuantumCircuit(1)
    qc.ry(x, 0)                       # the data point becomes a rotation angle
    return Statevector(qc)

def kernel_1q(x, x2):
    return abs(encode_1q(x).inner(encode_1q(x2))) ** 2

for x, x2 in [(0.5, 0.5), (0.5, 1.0), (0.5, 2.0), (0.0, np.pi)]:
    print(f"k({x:.2f}, {x2:.2f}) = {kernel_1q(x, x2):.3f}    formula cos²((x - x')/2) = {np.cos((x - x2) / 2) ** 2:.3f}")

## A two-qubit feature map
Two features, two qubits. Each feature sets a phase, and a CNOT–phase–CNOT block mixes them through an
entangling "ZZ" term. Repeating it twice gives the classic **ZZ feature map**.

In [ ]:
from qiskit.circuit import ParameterVector

x = ParameterVector("x", 2)
fmap = QuantumCircuit(2)
for repetition in range(2):
    fmap.h([0, 1])
    fmap.p(2 * x[0], 0)
    fmap.p(2 * x[1], 1)
    fmap.cx(0, 1)
    fmap.p(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)     # depends on BOTH features
    fmap.cx(0, 1)
fmap.draw("mpl", style="clifford", fold=-1)

## The kernel matrix
One statevector per data point, then all overlaps at once.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=120, noise=0.15, random_state=0)
X = MinMaxScaler((0, np.pi / 2)).fit_transform(X)            # features become angles between 0 and π/2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

def statevectors(data):
    return np.array([Statevector(fmap.assign_parameters(point)).data for point in data])

def quantum_kernel(A, B):
    return np.abs(statevectors(A).conj() @ statevectors(B).T) ** 2      # |<phi(a)|phi(b)>|^2 for every pair

K_train = quantum_kernel(X_train, X_train)
order = np.argsort(y_train)
plt.figure(figsize=(4.6, 3.9))
plt.imshow(K_train[np.ix_(order, order)], cmap="Purples", vmin=0, vmax=1)
plt.colorbar(label="k(x, x')")
plt.title("Quantum kernel matrix (sorted by class)")
plt.grid(False)
plt.show()

## A quantum SVM
Exactly week 4's recipe: hand the matrix to `SVC(kernel="precomputed")`. For new points we need their
kernel values against the *training* points.

In [ ]:
from sklearn.svm import SVC

K_test = quantum_kernel(X_test, X_train)
qsvm = SVC(kernel="precomputed").fit(K_train, y_train)
classical = SVC(kernel="rbf").fit(X_train, y_train)
print("quantum kernel SVM, test accuracy:", round(qsvm.score(K_test, y_test), 3))
print("classical RBF SVM, test accuracy :", round(classical.score(X_test, y_test), 3))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
gx, gy = np.meshgrid(np.linspace(0, np.pi / 2, 50), np.linspace(0, np.pi / 2, 50))
grid = np.c_[gx.ravel(), gy.ravel()]
decision = qsvm.decision_function(quantum_kernel(grid, X_train)).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(4.8, 4.2), layout="constrained")
ax.contourf(gx, gy, decision, levels=20, cmap=apricot_blue, alpha=0.55)
ax.contour(gx, gy, decision, levels=[0], colors="#2c2842", linewidths=2)
ax.scatter(X_train[:, 0], X_train[:, 1], c=np.where(y_train == 0, "#ff8a50", "#5577f5"), s=18)
ax.set_title("The boundary drawn by a quantum kernel")
plt.show()

## Your turn: a valid kernel, and its feature vectors
A kernel must be symmetric with no negative eigenvalues (ML week 4). Check both for `K_train`. Then check
where the "feature space" is: with $\rho(x) = |\varphi(x)\rangle\langle\varphi(x)|$, the kernel is the ordinary
inner product of two matrices, $k(x, x') = \mathrm{Tr}\big(\rho(x)\,\rho(x')\big)$. Verify it for two training points.

In [ ]:
eigenvalues = None        # TODO: np.linalg.eigvalsh(K_train)
print("symmetric:", np.allclose(K_train, K_train.T))
# TODO: print the smallest eigenvalue
# TODO: rho_a = np.outer(s_a, s_a.conj()) for the statevectors of X_train[0] and X_train[1]; compare np.trace(rho_a @ rho_b).real with K_train[0, 1]

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
eigenvalues = np.linalg.eigvalsh(K_train)
print("symmetric:", np.allclose(K_train, K_train.T), "  smallest eigenvalue:", f"{eigenvalues.min():.1e}", "(zero, up to rounding)")
s_a, s_b = statevectors(X_train[:2])
rho_a, rho_b = np.outer(s_a, s_a.conj()), np.outer(s_b, s_b.conj())
print(f"Tr(rho_a rho_b) = {np.trace(rho_a @ rho_b).real:.6f}   K_train[0, 1] = {K_train[0, 1]:.6f}")
print("Every quantum kernel of this kind is automatically valid: it's an inner product of density matrices,")
print("feature vectors with 4^n real entries (16 for our 2 qubits).")